**Training with NBC**

In [2]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
import warnings
warnings.filterwarnings("ignore")

X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label'] - 1


# Define the fitness function for BGWO
def fitness_function(position):
    # Select features based on the binary position vector
    selected_features = [i for i, bit in enumerate(position) if bit > 0.5]
    if not selected_features:  # Avoid empty feature subsets
        return 0.0
    
    model = GaussianNB()
    stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    X_sel = X.iloc[:, selected_features]
    scores = cross_val_score(model, X_sel, y, cv=stratified_kfold, scoring='accuracy')
    return scores.mean()

# Initialize BGWO parameters
num_wolves = 10
num_features = X.shape[1]  # Number of features in the dataset
max_iterations = 20
alpha, beta, delta = None, None, None

# Initialize the wolves' positions randomly (binary positions for feature selection)
wolves = np.random.randint(0, 2, size=(num_wolves, num_features))

for i in range(num_wolves):
    num_selected = np.random.randint(1, num_features + 1)
    selected = np.random.choice(num_features, num_selected, replace=False)
    wolves[i, selected] = 1

# BGWO algorithm
for iteration in range(max_iterations):
    fitness = np.array([fitness_function(wolf) for wolf in wolves])
    
    # Update alpha, beta, and delta wolves
    sorted_indices = np.argsort(fitness)[::-1]
    alpha, beta, delta = wolves[sorted_indices[:3]]
    
    a = 2 - 2 * iteration / max_iterations
    # Update positions of wolves
    for i in range(num_wolves):
        for j in range(num_features):
            # Calculate A and C for alpha, beta, delta
            A1 = 2 * a * np.random.rand() - a
            C1 = 2 * np.random.rand()
            D_alpha = abs(C1 * alpha[j] - wolves[i, j])
            X1 = alpha[j] - A1 * D_alpha
            
            A2 = 2 * a * np.random.rand() - a
            C2 = 2 * np.random.rand()
            D_beta = abs(C2 * beta[j] - wolves[i, j])
            X2 = beta[j] - A2 * D_beta
            
            A3 = 2 * a * np.random.rand() - a
            C3 = 2 * np.random.rand()
            D_delta = abs(C3 * delta[j] - wolves[i, j])
            X3 = delta[j] - A3 * D_delta
            
            # Update position with sigmoid transfer
            new_pos = (X1 + X2 + X3) / 3
            prob = 1 / (1 + np.exp(-new_pos))  # Sigmoid
            wolves[i, j] = 1 if np.random.rand() < prob else 0
# Evaluate the best solution
best_wolf = alpha
selected_features = [i for i in range(len(best_wolf)) if best_wolf[i] > 0.5]

if not selected_features:
    print(" No features selected by BGWO. Using all features as fallback.")
    selected_features = list(range(num_features))  # fallback: use all features

print("Number of Selected features: ", len(selected_features))
print("Selected features: ", selected_features)

X_selected = X.iloc[:, selected_features]
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42, stratify=y)


svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)

svm_acc = accuracy_score(y_test, svm_preds)
svm_percision = precision_score(y_test, svm_preds, average='macro')
svm_recall = recall_score(y_test, svm_preds, average='macro')
svm_f1 = f1_score(y_test, svm_preds, average='macro')

print("SVM")
print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")
print(f"SVM Percision on latent features selected by GA: {svm_percision * 100:.4f}")
print(f"SVM Recall on latent features selected by GA: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on latent features selected by GA: {svm_f1 * 100:.4f}")

nbc_model = GaussianNB()
nbc_model.fit(X_train, y_train)
nbc_preds = nbc_model.predict(X_test)

nbc_acc = accuracy_score(y_test, nbc_preds)
nbc_percision = precision_score(y_test, nbc_preds, average='macro')
nbc_recall = recall_score(y_test, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test, nbc_preds, average='macro')

print("NBC")
print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on latent features selected by GA: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on latent features selected by GA: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on latent features selected by GA: {nbc_f1 * 100:.4f}")

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_preds)
rf_percision = precision_score(y_test, rf_preds, average='macro')
rf_recall = recall_score(y_test, rf_preds, average='macro')
rf_f1 = f1_score(y_test, rf_preds, average='macro')

print("RF")
print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by GA: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by GA: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by GA: {rf_f1 * 100:.4f}")

Number of Selected features:  593
Selected features:  [1, 2, 5, 6, 7, 8, 9, 11, 12, 14, 16, 17, 18, 19, 21, 22, 24, 28, 30, 32, 33, 35, 36, 37, 39, 40, 41, 42, 43, 44, 46, 47, 48, 50, 51, 52, 54, 56, 57, 60, 61, 62, 63, 64, 65, 68, 69, 70, 71, 72, 73, 75, 76, 78, 81, 83, 84, 87, 88, 89, 91, 92, 93, 94, 97, 98, 99, 101, 103, 104, 105, 107, 109, 110, 112, 113, 114, 115, 117, 118, 119, 120, 122, 123, 125, 126, 127, 128, 129, 130, 131, 132, 134, 136, 138, 139, 140, 141, 142, 144, 145, 146, 148, 149, 150, 151, 152, 154, 155, 159, 160, 162, 163, 165, 166, 167, 169, 170, 172, 174, 175, 178, 179, 180, 189, 190, 191, 192, 194, 196, 197, 198, 199, 202, 203, 204, 205, 207, 208, 209, 210, 211, 212, 213, 215, 216, 218, 219, 220, 221, 224, 225, 227, 231, 232, 233, 234, 235, 237, 241, 243, 245, 246, 250, 251, 252, 253, 254, 255, 256, 257, 259, 260, 261, 263, 266, 267, 269, 271, 272, 275, 276, 277, 278, 280, 282, 283, 284, 285, 287, 288, 292, 295, 296, 298, 299, 300, 301, 302, 303, 304, 305, 306, 308,

**Training with SVM**

In [3]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
import warnings
warnings.filterwarnings("ignore")

X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label'] - 1


# Define the fitness function for BGWO
def fitness_function(position):
    # Select features based on the binary position vector
    selected_features = [i for i, bit in enumerate(position) if bit > 0.5]
    if not selected_features:  # Avoid empty feature subsets
        return 0.0
    
    model = SVC(random_state=42)
    stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    X_sel = X.iloc[:, selected_features]
    scores = cross_val_score(model, X_sel, y, cv=stratified_kfold, scoring='accuracy')
    return scores.mean()

# Initialize BGWO parameters
num_wolves = 10
num_features = X.shape[1]  # Number of features in the dataset
max_iterations = 20
alpha, beta, delta = None, None, None

# Initialize the wolves' positions randomly (binary positions for feature selection)
wolves = np.random.randint(0, 2, size=(num_wolves, num_features))

for i in range(num_wolves):
    num_selected = np.random.randint(1, num_features + 1)
    selected = np.random.choice(num_features, num_selected, replace=False)
    wolves[i, selected] = 1

# BGWO algorithm
for iteration in range(max_iterations):
    fitness = np.array([fitness_function(wolf) for wolf in wolves])
    
    # Update alpha, beta, and delta wolves
    sorted_indices = np.argsort(fitness)[::-1]
    alpha, beta, delta = wolves[sorted_indices[:3]]
    
    a = 2 - 2 * iteration / max_iterations
    # Update positions of wolves
    for i in range(num_wolves):
        for j in range(num_features):
            # Calculate A and C for alpha, beta, delta
            A1 = 2 * a * np.random.rand() - a
            C1 = 2 * np.random.rand()
            D_alpha = abs(C1 * alpha[j] - wolves[i, j])
            X1 = alpha[j] - A1 * D_alpha
            
            A2 = 2 * a * np.random.rand() - a
            C2 = 2 * np.random.rand()
            D_beta = abs(C2 * beta[j] - wolves[i, j])
            X2 = beta[j] - A2 * D_beta
            
            A3 = 2 * a * np.random.rand() - a
            C3 = 2 * np.random.rand()
            D_delta = abs(C3 * delta[j] - wolves[i, j])
            X3 = delta[j] - A3 * D_delta
            
            # Update position with sigmoid transfer
            new_pos = (X1 + X2 + X3) / 3
            prob = 1 / (1 + np.exp(-new_pos))  # Sigmoid
            wolves[i, j] = 1 if np.random.rand() < prob else 0
# Evaluate the best solution
best_wolf = alpha
selected_features = [i for i in range(len(best_wolf)) if best_wolf[i] > 0.5]

if not selected_features:
    print(" No features selected by BGWO. Using all features as fallback.")
    selected_features = list(range(num_features))  # fallback: use all features

print("Number of Selected features: ", len(selected_features))
print("Selected features: ", selected_features)

X_selected = X.iloc[:, selected_features]
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42, stratify=y)


svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)

svm_acc = accuracy_score(y_test, svm_preds)
svm_percision = precision_score(y_test, svm_preds, average='macro')
svm_recall = recall_score(y_test, svm_preds, average='macro')
svm_f1 = f1_score(y_test, svm_preds, average='macro')

print("SVM")
print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")
print(f"SVM Percision on latent features selected by GA: {svm_percision * 100:.4f}")
print(f"SVM Recall on latent features selected by GA: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on latent features selected by GA: {svm_f1 * 100:.4f}")

nbc_model = GaussianNB()
nbc_model.fit(X_train, y_train)
nbc_preds = nbc_model.predict(X_test)

nbc_acc = accuracy_score(y_test, nbc_preds)
nbc_percision = precision_score(y_test, nbc_preds, average='macro')
nbc_recall = recall_score(y_test, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test, nbc_preds, average='macro')

print("NBC")
print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on latent features selected by GA: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on latent features selected by GA: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on latent features selected by GA: {nbc_f1 * 100:.4f}")

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_preds)
rf_percision = precision_score(y_test, rf_preds, average='macro')
rf_recall = recall_score(y_test, rf_preds, average='macro')
rf_f1 = f1_score(y_test, rf_preds, average='macro')

print("RF")
print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by GA: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by GA: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by GA: {rf_f1 * 100:.4f}")

Number of Selected features:  584
Selected features:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 11, 13, 15, 16, 17, 18, 24, 28, 29, 32, 33, 34, 35, 36, 38, 39, 41, 42, 43, 44, 47, 48, 50, 51, 54, 55, 56, 58, 59, 60, 61, 62, 63, 65, 66, 68, 69, 70, 71, 72, 73, 75, 77, 79, 81, 82, 85, 86, 87, 88, 90, 91, 94, 96, 97, 99, 101, 106, 109, 110, 111, 114, 116, 117, 118, 119, 120, 122, 125, 127, 128, 129, 130, 131, 133, 134, 138, 140, 141, 142, 144, 145, 146, 147, 148, 149, 150, 151, 153, 154, 155, 156, 157, 158, 160, 161, 164, 165, 166, 168, 169, 170, 171, 172, 173, 175, 176, 177, 178, 179, 180, 182, 183, 185, 187, 189, 190, 191, 192, 196, 198, 200, 202, 205, 206, 207, 208, 209, 210, 211, 212, 214, 215, 217, 220, 221, 222, 224, 226, 227, 229, 233, 234, 236, 237, 238, 239, 241, 242, 243, 245, 246, 248, 249, 250, 251, 256, 257, 258, 260, 262, 267, 272, 273, 276, 277, 278, 279, 280, 281, 283, 284, 285, 286, 287, 289, 291, 292, 294, 296, 297, 299, 300, 301, 302, 304, 305, 306, 308, 309, 310, 311, 314, 315, 316,

**Training with RF**

In [4]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
import warnings
warnings.filterwarnings("ignore")

X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label'] - 1


# Define the fitness function for BGWO
def fitness_function(position):
    # Select features based on the binary position vector
    selected_features = [i for i, bit in enumerate(position) if bit > 0.5]
    if not selected_features:  # Avoid empty feature subsets
        return 0.0
    
    model = RandomForestClassifier(random_state=42)
    stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    X_sel = X.iloc[:, selected_features]
    scores = cross_val_score(model, X_sel, y, cv=stratified_kfold, scoring='accuracy')
    return scores.mean()

# Initialize BGWO parameters
num_wolves = 10
num_features = X.shape[1]  # Number of features in the dataset
max_iterations = 20
alpha, beta, delta = None, None, None

# Initialize the wolves' positions randomly (binary positions for feature selection)
wolves = np.random.randint(0, 2, size=(num_wolves, num_features))

for i in range(num_wolves):
    num_selected = np.random.randint(1, num_features + 1)
    selected = np.random.choice(num_features, num_selected, replace=False)
    wolves[i, selected] = 1

# BGWO algorithm
for iteration in range(max_iterations):
    print('Iteration: ', iteration)
    fitness = np.array([fitness_function(wolf) for wolf in wolves])
    
    # Update alpha, beta, and delta wolves
    sorted_indices = np.argsort(fitness)[::-1]
    alpha, beta, delta = wolves[sorted_indices[:3]]
    
    a = 2 - 2 * iteration / max_iterations
    # Update positions of wolves
    for i in range(num_wolves):
        for j in range(num_features):
            # Calculate A and C for alpha, beta, delta
            A1 = 2 * a * np.random.rand() - a
            C1 = 2 * np.random.rand()
            D_alpha = abs(C1 * alpha[j] - wolves[i, j])
            X1 = alpha[j] - A1 * D_alpha
            
            A2 = 2 * a * np.random.rand() - a
            C2 = 2 * np.random.rand()
            D_beta = abs(C2 * beta[j] - wolves[i, j])
            X2 = beta[j] - A2 * D_beta
            
            A3 = 2 * a * np.random.rand() - a
            C3 = 2 * np.random.rand()
            D_delta = abs(C3 * delta[j] - wolves[i, j])
            X3 = delta[j] - A3 * D_delta
            
            # Update position with sigmoid transfer
            new_pos = (X1 + X2 + X3) / 3
            prob = 1 / (1 + np.exp(-new_pos))  # Sigmoid
            wolves[i, j] = 1 if np.random.rand() < prob else 0
# Evaluate the best solution
best_wolf = alpha
selected_features = [i for i in range(len(best_wolf)) if best_wolf[i] > 0.5]

if not selected_features:
    print(" No features selected by BGWO. Using all features as fallback.")
    selected_features = list(range(num_features))  # fallback: use all features

print("Number of Selected features: ", len(selected_features))
print("Selected features: ", selected_features)

X_selected = X.iloc[:, selected_features]
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42, stratify=y)


svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)

svm_acc = accuracy_score(y_test, svm_preds)
svm_percision = precision_score(y_test, svm_preds, average='macro')
svm_recall = recall_score(y_test, svm_preds, average='macro')
svm_f1 = f1_score(y_test, svm_preds, average='macro')

print("SVM")
print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")
print(f"SVM Percision on latent features selected by GA: {svm_percision * 100:.4f}")
print(f"SVM Recall on latent features selected by GA: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on latent features selected by GA: {svm_f1 * 100:.4f}")

nbc_model = GaussianNB()
nbc_model.fit(X_train, y_train)
nbc_preds = nbc_model.predict(X_test)

nbc_acc = accuracy_score(y_test, nbc_preds)
nbc_percision = precision_score(y_test, nbc_preds, average='macro')
nbc_recall = recall_score(y_test, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test, nbc_preds, average='macro')

print("NBC")
print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on latent features selected by GA: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on latent features selected by GA: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on latent features selected by GA: {nbc_f1 * 100:.4f}")

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_preds)
rf_percision = precision_score(y_test, rf_preds, average='macro')
rf_recall = recall_score(y_test, rf_preds, average='macro')
rf_f1 = f1_score(y_test, rf_preds, average='macro')

print("RF")
print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by GA: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by GA: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by GA: {rf_f1 * 100:.4f}")

Iteration:  0
Iteration:  1
Iteration:  2
Iteration:  3
Iteration:  4
Iteration:  5
Iteration:  6
Iteration:  7
Iteration:  8
Iteration:  9
Iteration:  10
Iteration:  11
Iteration:  12
Iteration:  13
Iteration:  14
Iteration:  15
Iteration:  16
Iteration:  17
Iteration:  18
Iteration:  19
Number of Selected features:  590
Selected features:  [0, 1, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 16, 17, 19, 21, 22, 23, 24, 25, 26, 27, 29, 33, 35, 37, 38, 39, 40, 41, 42, 46, 47, 49, 50, 52, 53, 54, 55, 57, 58, 60, 64, 65, 66, 67, 69, 70, 71, 72, 73, 75, 76, 77, 80, 81, 82, 85, 87, 88, 89, 95, 97, 99, 101, 107, 109, 110, 111, 112, 114, 115, 116, 117, 118, 119, 123, 124, 125, 126, 127, 128, 131, 132, 133, 134, 135, 136, 137, 138, 141, 144, 147, 148, 150, 151, 152, 153, 154, 155, 156, 158, 159, 160, 161, 162, 163, 164, 165, 166, 168, 170, 173, 174, 176, 177, 180, 182, 184, 186, 187, 188, 189, 190, 191, 192, 193, 194, 197, 198, 199, 200, 202, 204, 205, 206, 207, 208, 209, 210, 211, 212, 214, 215, 216, 21